# 🥉 Module 1: Bronze Layer - Data Ingestion

## Overview

In this notebook, we'll download raw trip data from Oslo City Bike and store it in the Bronze layer of our Lakehouse.

**What you'll learn:**
- Download data from external HTTP sources
- Store raw files in the Lakehouse
- Add metadata for data lineage

---

**Prerequisites:**
- This notebook must be attached to your `oslo_bysykkel_lakehouse`
- The folder `Files/bronze/trips/` should exist (created in Step 1)

## Step 1: Configuration

First, let's define our configuration variables.

In [ ]:
# Configuration
YEAR = 2026
MONTHS = list(range(1, 9))  # January to August (available months)

# Base URL for Oslo Bysykkel trip data
BASE_URL = "https://data.urbansharing.com/oslobysykkel.no/trips/v1"

# Lakehouse paths
BRONZE_PATH = "/lakehouse/default/Files/bronze/trips"

print(f"Configuration:")
print(f"  Year: {YEAR}")
print(f"  Months: {MONTHS}")
print(f"  Base URL: {BASE_URL}")
print(f"  Bronze Path: {BRONZE_PATH}")

## Step 2: Create Folder Structure

Ensure the Bronze folder structure exists.

In [ ]:
import os

# Create the bronze/trips directory if it doesn't exist
os.makedirs(BRONZE_PATH, exist_ok=True)

print(f"✅ Bronze folder ready: {BRONZE_PATH}")

## Step 3: Download Raw CSV Files

Now we'll download all monthly CSV files from Oslo City Bike.

**Important:** This downloads the data exactly as provided - no transformations!

In [ ]:
import requests
from datetime import datetime

# Track ingestion metadata
ingestion_log = []

def download_file(year, month):
    """
    Download a single month's trip data from Oslo Bysykkel.
    
    Args:
        year: The year (e.g., 2026)
        month: The month number (1-12)
    
    Returns:
        dict: Metadata about the download
    """
    # Format month with leading zero
    month_str = f"{month:02d}"
    
    # Build URL and file paths
    source_url = f"{BASE_URL}/{year}/{month_str}.csv"
    file_name = f"{year}_{month_str}.csv"
    file_path = f"{BRONZE_PATH}/{file_name}"
    
    print(f"Downloading: {source_url}")
    
    try:
        # Download the file
        response = requests.get(source_url, timeout=60)
        response.raise_for_status()
        
        # Save to Bronze layer
        with open(file_path, 'wb') as f:
            f.write(response.content)
        
        # Count rows (subtract 1 for header)
        row_count = response.text.count('\n') - 1
        
        # Create metadata record
        metadata = {
            'source_file': file_name,
            'source_url': source_url,
            'ingestion_time': datetime.now().isoformat(),
            'file_size_bytes': len(response.content),
            'row_count': row_count,
            'status': 'SUCCESS'
        }
        
        print(f"  ✅ Saved: {file_name} ({row_count:,} rows, {len(response.content):,} bytes)")
        return metadata
        
    except Exception as e:
        print(f"  ❌ Error: {str(e)}")
        return {
            'source_file': file_name,
            'source_url': source_url,
            'ingestion_time': datetime.now().isoformat(),
            'file_size_bytes': 0,
            'row_count': 0,
            'status': f'FAILED: {str(e)}'
        }

# Download all months
print(f"\n{'='*60}")
print(f"Starting download of {len(MONTHS)} files...")
print(f"{'='*60}\n")

for month in MONTHS:
    metadata = download_file(YEAR, month)
    ingestion_log.append(metadata)

print(f"\n{'='*60}")
print("Download complete!")
print(f"{'='*60}")

## Step 4: Save Ingestion Log

For data lineage and auditing, we save a log of all ingested files.

In [ ]:
import csv

# Save ingestion log as CSV
log_path = f"{BRONZE_PATH}/_ingestion_log.csv"

with open(log_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['source_file', 'source_url', 'ingestion_time', 
                                            'file_size_bytes', 'row_count', 'status'])
    writer.writeheader()
    writer.writerows(ingestion_log)

print(f"✅ Ingestion log saved to: {log_path}")

## Step 5: Verify the Downloads

Let's verify that all files were downloaded successfully.

In [ ]:
# List all files in the Bronze folder
print("Files in Bronze layer:")
print("-" * 50)

total_size = 0
total_rows = 0

for entry in ingestion_log:
    if entry['status'] == 'SUCCESS':
        print(f"  📄 {entry['source_file']:15} | {entry['row_count']:>10,} rows | {entry['file_size_bytes']:>12,} bytes")
        total_size += entry['file_size_bytes']
        total_rows += entry['row_count']

print("-" * 50)
print(f"  {'TOTAL':15} | {total_rows:>10,} rows | {total_size:>12,} bytes")
print(f"\n  Total size: {total_size / (1024*1024):.2f} MB")

## Step 6: Bronze Ingestion Complete

The Bronze layer is now stored as raw CSV files in `Files/bronze/trips/`. No table is created at this stage.

The SQL analytics endpoint cannot query CSV files in the Lakehouse Files area as `dbo` tables. SQL endpoint exercises begin after the Silver notebook creates a Delta table.

## ✅ Module Complete!

### Summary

In this module, you have:

1. ✅ Downloaded all 2026 monthly trip data files
2. ✅ Stored raw CSV files in the Bronze layer
3. ✅ Created an ingestion log for data lineage
4. ✅ Verified the downloaded files and their row counts

### Key Takeaways

- **Bronze layer stores raw data** - No business transformations are applied
- **Metadata is essential** - We tracked source, time, and size
- **CSV files are not SQL endpoint tables** - The SQL analytics endpoint becomes available for the Delta tables created in the Silver and Gold layers

### Next Step

Continue to **Module 2: Silver Layer** where the raw CSV files are read, cleaned, typed, and written as the `silver_trips` Delta table.